## Part 1 RDDs
Repeat the steps of Assignment 1, i.e. calculation of chi-square values and output of the sorted top terms per category, as well as the joined dictionary, using RDDs and transformations. Write the output to a file output_rdd.txt. Compare the generated output_rdd.txt with your generated output.txt from Assignment 1 and describe your observations briefly in the submission report (see Part 3).

In [14]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

from pyspark.sql.functions import explode, split, lower


import json
import re
from datetime import datetime

# Stop existing SparkContext if it's running
if SparkContext._active_spark_context:
    SparkContext._active_spark_context.stop()



In [15]:
# Initialize Spark context and session
conf = SparkConf().setAppName("Part1")
sc = SparkContext(conf=conf)
spark = SparkSession(sc)

25/05/13 16:09:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/05/13 16:09:13 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/05/13 16:09:13 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/05/13 16:09:13 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/05/13 16:09:13 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/05/13 16:09:13 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
25/05/13 16:09:13 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
25/05/13 16:09:13 WARN Utils: Service 'SparkUI' could not bind on port 4047. Attempting port 4048.
25/05/13 16:09:13 WARN Utils: Service 'SparkUI' could not bind on port 4048. Attempting port 4049.
25/05/13 16:09:13 WARN Utils: Service 'SparkUI' could not bind on port 4049. Attempting port 4050.
25/05/13 1

In [16]:
spark

### Preprocessing

In [17]:
# Define the stopwords file 
stop_file = "stopwords.txt"

# Load stopwords into a set
with open(stop_file, "r") as f:
    stopwords = set(f.read().strip().split())
    
# Load and preprocess the Amazon reviews dataset (devset file)
input_file = "hdfs:///user/dic25_shared/amazon-reviews/full/reviews_devset.json"
#input_file = "hdfs:///user/dic25_shared/amazon-reviews/full/reviewscombined.json"
reviews_rdd = sc.textFile(input_file)

In [18]:
# Show first three objects of reviews_rdd
reviews_rdd.take(3)

['{"reviewerID": "A2VNYWOPJ13AFP", "asin": "0981850006", "reviewerName": "Amazon Customer \\"carringt0n\\"", "helpful": [6, 7], "reviewText": "This was a gift for my other husband.  He\'s making us things from it all the time and we love the food.  Directions are simple, easy to read and interpret, and fun to make.  We all love different kinds of cuisine and Raichlen provides recipes from everywhere along the barbecue trail as he calls it. Get it and just open a page.  Have at it.  You\'ll love the food and it has provided us with an insight into the culture that produced it. It\'s all about broadening horizons.  Yum!!", "overall": 5.0, "summary": "Delish", "unixReviewTime": 1259798400, "reviewTime": "12 3, 2009", "category": "Patio_Lawn_and_Garde"}',
 '{"reviewerID": "A2E5XXXC07AGA7", "asin": "B00002N66D", "reviewerName": "James", "helpful": [1, 1], "reviewText": "This is a very nice spreader.  It feels very solid and the pneumatic tires give it great maneuverability and handling over

In [19]:
# Preprocess the text and check for valid words
def preprocess_text(text):
    text = text.lower()
    unigrams = re.split(r'\s+|\d+|[(){}[\].!?,;:+=_"\'`~#@&*%€$§\\/\-]', text)
    unigrams = set(unigrams)
    return unigrams

def valid_word(word):
    if len(word) > 1 and word not in stopwords:
        return word
    
json_rdd = reviews_rdd.map(lambda line: json.loads(line))

word_category_rdd = json_rdd.flatMap(lambda x: [(x["category"], word) for word in preprocess_text(x['reviewText']) if valid_word(word)])

In [20]:
word_category_rdd.take(10)

[('Patio_Lawn_and_Garde', 'insight'),
 ('Patio_Lawn_and_Garde', 'things'),
 ('Patio_Lawn_and_Garde', 'raichlen'),
 ('Patio_Lawn_and_Garde', 'open'),
 ('Patio_Lawn_and_Garde', 'horizons'),
 ('Patio_Lawn_and_Garde', 'food'),
 ('Patio_Lawn_and_Garde', 'interpret'),
 ('Patio_Lawn_and_Garde', 'make'),
 ('Patio_Lawn_and_Garde', 'broadening'),
 ('Patio_Lawn_and_Garde', 'cuisine')]

In [21]:
combined_rdd = word_category_rdd.map(lambda x: (x,1)).reduceByKey(lambda x,y: x + y)

In [22]:
combined_rdd.take(10)

[(('Patio_Lawn_and_Garde', 'horizons'), 1),
 (('Patio_Lawn_and_Garde', 'food'), 23),
 (('Patio_Lawn_and_Garde', 'interpret'), 1),
 (('Patio_Lawn_and_Garde', 'broadening'), 1),
 (('Patio_Lawn_and_Garde', 'cuisine'), 1),
 (('Patio_Lawn_and_Garde', 'easy'), 142),
 (('Patio_Lawn_and_Garde', 'trail'), 4),
 (('Patio_Lawn_and_Garde', 'provided'), 8),
 (('Patio_Lawn_and_Garde', 'time'), 141),
 (('Patio_Lawn_and_Garde', 'kinds'), 5)]

In [23]:
reduced_rdd = combined_rdd.map(lambda x: (x[0][0], (x[0][1],x[1])))

In [24]:
# Showing intermediate results
for category, word_count in reduced_rdd.groupByKey().take(2):
    print(category)
    word_count_list = list(word_count)
    for word, count_in_count in word_count_list[:5]:
        print(word, count_in_count)

[Stage 5:=============================>                             (1 + 1) / 2]

Kindle_Store
outcome 8
immediately 41
bad 133
time 457
separate 5
Electronic
purchasing 89
future 69
great 2223
nook 18
stuff 90


### Intermediate step


Before calculating chi-square statistics we first need to explore the how the reviews are distributed across the categories which helps us to understand how many documents belong to a certain category.

In [25]:
# count dataset length
dataset_len = reviews_rdd.count()
dataset_len

78829

In [26]:
reviews_per_category_count_rdd = json_rdd.map(lambda x: (x['category'],1)).reduceByKey(lambda x, y: x + y)
reviews_per_category_count_rdd.collect()

[('Apps_for_Android', 2638),
 ('Book', 22507),
 ('Toys_and_Game', 2253),
 ('Office_Product', 1243),
 ('Digital_Music', 836),
 ('Automotive', 1374),
 ('Beauty', 2023),
 ('Kindle_Store', 3205),
 ('Electronic', 7825),
 ('Movies_and_TV', 4607),
 ('Tools_and_Home_Improvement', 1926),
 ('Grocery_and_Gourmet_Food', 1297),
 ('Musical_Instrument', 500),
 ('CDs_and_Vinyl', 3749),
 ('Clothing_Shoes_and_Jewelry', 5749),
 ('Home_and_Kitche', 4254),
 ('Cell_Phones_and_Accessorie', 3447),
 ('Pet_Supplie', 1235),
 ('Baby', 916),
 ('Health_and_Personal_Care', 2982),
 ('Patio_Lawn_and_Garde', 994),
 ('Sports_and_Outdoor', 3269)]

### Calculate Chi-Square

Next, we prepare the values A, B, C and D which are needed to compute the chi-square. 

In [27]:
start_time = datetime.now()


In [28]:
# calculate value A (number of documents in c which contain t)
A_value_rdd = reduced_rdd.map(lambda x: ((x[0], x[1][0]), x[1][1]))
A_value_rdd.take(1)

[(('Patio_Lawn_and_Garde', 'horizons'), 1)]

In [29]:
# calculate how often a word occurs across all categories
B_value_complete_rdd = reduced_rdd.map(lambda x: (x[1][0],x[1][1])).reduceByKey(lambda x, y: x + y)
B_value_complete_rdd.take(1)

[('fairly', 723)]

In [30]:
# calulate B value (number of documents not in c which contain t) value is obtained by subtracting A from total count
# the value for the join is saved in a combined rdd in order to save compute power later on
A_B_value_rdd = A_value_rdd.map(lambda x: (x[0][1], (x[0][0], x[1]))).join(B_value_complete_rdd).map(lambda x: ((x[1][0][0], x[0]),  (x[1][0][1], x[1][1] - x[1][0][1])))
A_B_value_rdd.take(1)

[(('Patio_Lawn_and_Garde', 'insight'), (1, 428))]

In [31]:
# calculate C value (number of documents in c without t) occurences of word per category subtracted from all documents in category
C_value_rdd = reduced_rdd.join(reviews_per_category_count_rdd).map(lambda x: ((x[0], x[1][0][0]), x[1][1] - x[1][0][1]))
C_value_rdd.take(1)

[(('Electronic', 'cable'), 7262)]

In [32]:
# calculate D value (number of documents not in c without t) add up A, B and C and subtract from total number of documents
# all values are saved in combined RDD to save computational resources
A_B_C_D_values_rdd = A_B_value_rdd.join(C_value_rdd).map(lambda x: (x[0], (x[1][0][0], x[1][0][1], x[1][1], dataset_length - (x[1][0][0] + x[1][0][1] + x[1][1]))))
A_B_C_D_values_rdd.take(1)                                         

[(('Clothing_Shoes_and_Jewelry', 'clip'), (15, 325, 5734, 78822802))]

In [33]:
# caluclate chi square
chi_square_rdd = A_B_C_D_values_rdd.map(lambda x: (x[0][0], (x[0][1],
                                                             ((dataset_length * (x[1][0] * x[1][3] - x[1][1] * x[1][2])** 2)/
                                                              ((x[1][0] + x[1][1]) * (x[1][0] + x[1][2])
                                                               * (x[1][1] + x[1][3]) * (x[1][2] + x[1][3]))))))

In [34]:
chi_square_rdd.take(1)

[('Clothing_Shoes_and_Jewelry', ('clip', 9044.678546325938))]

In [35]:
# sort values by category and chi square value
chi_sort_rdd = chi_square_rdd.sortBy(lambda x: (x[0], x[1][1]), ascending=False)
chi_sort_rdd.take(10)

[('Toys_and_Game', ('loves', 1766128.875491677)),
 ('Toys_and_Game', ('son', 1644150.3374810684)),
 ('Toys_and_Game', ('toys', 1575528.2428240855)),
 ('Toys_and_Game', ('play', 1300788.556974604)),
 ('Toys_and_Game', ('lego', 1268452.2611462213)),
 ('Toys_and_Game', ('year', 1197850.2005178037)),
 ('Toys_and_Game', ('kids', 1169917.0198615466)),
 ('Toys_and_Game', ('grandson', 1108279.235702337)),
 ('Toys_and_Game', ('daughter', 1039132.7771135495)),
 ('Toys_and_Game', ('christmas', 1029560.8162340972))]

In [36]:
# sort values by category and select top 75 chi square values
chi_cropped_rdd = chi_sort_rdd.groupByKey().map(lambda x: (x[0], list(x[1])[:75])).sortByKey()
chi_cropped_rdd.take(1)

[('Apps_for_Android',
  [('games', 3504879.3797522974),
   ('play', 2857079.257628206),
   ('kindle', 1918251.0265580453),
   ('graphics', 1676284.376349798),
   ('addictive', 1424900.250403611),
   ('fire', 1233864.8251190516),
   ('challenging', 1205864.6165487904),
   ('coins', 1077053.3063619367),
   ('addicting', 1069317.3610370276),
   ('playing', 1023413.6786970634),
   ('levels', 1010525.5997162092),
   ('free', 882504.5289997266),
   ('love', 733520.0352194578),
   ('ads', 705444.5078549781),
   ('puzzles', 666691.4945684309),
   ('apps', 657862.2912569465),
   ('bingo', 438241.07314563345),
   ('download', 420629.0976707881),
   ('awesome', 388585.8788421339),
   ('great', 381236.6604948302),
   ('downloaded', 351114.807339682),
   ('time', 346062.54850114894),
   ('played', 338568.2824891197),
   ('mahjong', 336656.5474157867),
   ('facebook', 335425.68271610205),
   ('hints', 299985.788471718),
   ('faotd', 298810.6444263329),
   ('android', 274772.57308308873),
   ('puzzle

In [37]:
# collect RDD and write data to file
data = chi_cropped_rdd.collect()

# collect unique words from top 75
def extract_words(record):
    category, word_list = record
    return [word for word, _ in word_list]

words = chi_cropped_rdd.flatMap(extract_words)
words = sorted(set(words.collect()))

# stop time 
end_time = datetime.now()

# write data file
with open("output_rdd.txt", "w") as writer:
    for row in data:
        writer.write(str(row) + '\n')
    for word in words:
        writer.write(word + " ")

In [38]:
print("start time:", start_time)
print("end time:", end_time)
print("time elapsed:", end_time - start_time)

start time: 2025-05-13 16:09:50.959303
end time: 2025-05-13 16:10:11.960696
time elapsed: 0:00:21.001393


In [39]:
spark.stop()